# Data Projection

**Overview**
- 3D visual of a neural network using Tensor

**Setup**
- imports libaries

In [18]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from torch.utils.tensorboard import SummaryWriter

**Data**
- loads in the dataset

In [ ]:
batch_size=64
transform = transforms.Compose([transforms.ToTensor()])
trainset = torchvision.datasets.MNIST('data', train=True, download=True, transform=transform)
testset = torchvision.datasets.MNIST('data', train=False, download=True, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size, shuffle=True)
testloader = torch.utils.data.DataLoader(testset, batch_size, shuffle=False)

**Model**
- creates convolutional neural network

In [20]:
class SimpleNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 10, 5)
        self.conv2 = nn.Conv2d(10, 20, 5)
        self.fc1 = nn.Linear(320, 50)
        self.fc2 = nn.Linear(50, 10)

    def forward(self, x):
        x = F.relu(F.max_pool2d(self.conv1(x), 2))
        x = F.relu(F.max_pool2d(self.conv2(x), 2))
        x = x.view(-1, 320)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return F.log_softmax(x, dim=1)

net = SimpleNet()
optimizer = torch.optim.SGD(net.parameters(), lr=0.01, momentum=0.9)
criterion = nn.CrossEntropyLoss()

**Training**
- goes through dataset twice for training with dataset

In [21]:
for epoch in range(2):
    for i, (inputs, labels) in enumerate(trainloader):
        optimizer.zero_grad()
        outputs = net(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        if i % 500 == 0:
            print(f'Epoch {epoch}, Batch {i}, Loss: {loss.item():.3f}')

Epoch 0, Batch 0, Loss: 2.320
Epoch 0, Batch 500, Loss: 0.180
Epoch 1, Batch 0, Loss: 0.070
Epoch 1, Batch 500, Loss: 0.007


**Tensor**
- creates visualization for tensorboard

In [22]:
writer = SummaryWriter('runs/mnist')
all_embeddings = []
all_labels = []
all_images = []

with torch.no_grad():
    for images, labels in testloader:
        outputs = net(images)
        all_embeddings.append(outputs)
        all_labels.extend([str(x.item()) for x in labels])
        all_images.append(images)
        
        if len(all_embeddings) > 20:
            break

embeddings = torch.cat(all_embeddings)
images = torch.cat(all_images)

writer.add_embedding(embeddings, metadata=all_labels, label_img=images)
writer.close()

print("type \"tensorboard --logdir=runs\" inside of the terminal")
print("Go to http://localhost:6006 inside of your browser")

type "tensorboard --logdir=runs" inside of the terminal
Go to http://localhost:6006 inside of your browser
